# Aula05.Ex01 - Exemplo - Cubo - Transformação Geométrica 3D

### Primeiro, vamos importar as bibliotecas necessárias.

In [1]:
import glfw
from OpenGL.GL import *
import OpenGL.GL.shaders
import numpy as np
import glm
import math

### Inicializando janela

In [2]:
glfw.init()
glfw.window_hint(glfw.VISIBLE, glfw.FALSE);
window = glfw.create_window(700, 700, "Academia", None, None)

if (window == None):
    print("Failed to create GLFW window")
    glfwTerminate()
    
glfw.make_context_current(window)

### Shaders

Note que agora usamos vec3, já que estamos em 3D.

In [3]:
vertex_code = """
        attribute vec3 position;
        uniform mat4 mat_transformation;
        void main(){
            gl_Position = mat_transformation * vec4(position,1.0);
        }
        """

In [4]:
fragment_code = """
        uniform vec4 color;
        void main(){
            gl_FragColor = color;
        }
        """

### Requisitando slot para a GPU para nossos programas Vertex e Fragment Shaders

In [5]:
# Request a program and shader slots from GPU
program  = glCreateProgram()
vertex   = glCreateShader(GL_VERTEX_SHADER)
fragment = glCreateShader(GL_FRAGMENT_SHADER)


### Associando nosso código-fonte aos slots solicitados

In [6]:
# Set shaders source
glShaderSource(vertex, vertex_code)
glShaderSource(fragment, fragment_code)

### Compilando o Vertex Shader

Se há algum erro em nosso programa Vertex Shader, nosso app para por aqui.

In [7]:
# Compile shaders
glCompileShader(vertex)
if not glGetShaderiv(vertex, GL_COMPILE_STATUS):
    error = glGetShaderInfoLog(vertex).decode()
    print(error)
    raise RuntimeError("Erro de compilacao do Vertex Shader")


### Compilando o Fragment Shader

Se há algum erro em nosso programa Fragment Shader, nosso app para por aqui.

In [8]:
glCompileShader(fragment)
if not glGetShaderiv(fragment, GL_COMPILE_STATUS):
    error = glGetShaderInfoLog(fragment).decode()
    print(error)
    raise RuntimeError("Erro de compilacao do Fragment Shader")

### Associando os programas compilado ao programa principal

In [9]:
# Attach shader objects to the program
glAttachShader(program, vertex)
glAttachShader(program, fragment)


### Linkagem do programa

In [10]:
# Build program
glLinkProgram(program)
if not glGetProgramiv(program, GL_LINK_STATUS):
    print(glGetProgramInfoLog(program))
    raise RuntimeError('Linking error')
    
# Make program the default program
glUseProgram(program)

### Preparando dados para enviar a GPU

Até aqui, compilamos nossos Shaders para que a GPU possa processá-los.

Por outro lado, as informações de vértices geralmente estão na CPU e devem ser transmitidas para a GPU.


In [11]:
# preparando espaço para  vértices usando 3 coordenadas (x,y,z)

### ÍNDICES ###
'''
Cômodo: 0-6
Assento: 7-30
'''

#### ASSENTO - PARALELEPÍPEDO ###
#(largura, altura, profundidade)
w = 0.2
h = 0.05
d = 0.2

vertices = np.zeros(30, [("position", np.float32, 3)])

# preenchendo as coordenadas de cada vértice
vertices['position'] = [

    ### CÔMODO ###
    #Linha 1 - horizontal
    (0.0, 0.0, 0.0),
    (1.0, -0.3, -0.0),

    #Linha 2 - horizontal
    (0.0, 0.0, 0.0),
    (-1.0, -0.5, 0.0),

    #Linha 3 - vertical
    (0.0, 0.0,0.0),
    (0.0, 1.0, 0.0),

    ### BANCO ###

    #### ASSENTO - PARALELEPÍPEDO ###
    #(largura, altura, profundidade)

    # Face 1 do Cubo (frente)
    (-w, -h, +d),
    (+w, -h, +d),
    (-w, +h, +d),
    (+w, +h, +d),

    # Face 2 do Cubo (direita - lateral) 
    (+w, -h, +d),
    (+w, -h, -d),         
    (+w, +h, +d),
    (+w, +h, -d),
    
    # Face 3 do Cubo (Trás)
    (+w, -h, -d),
    (-w, -h, -d),            
    (+w, +h, -d),
    (-w, +h, -d),

    # Face 4 do Cubo (esquerda - lateral)
    (-w, -h, -d),
    (-w, -h, +d),         
    (-w, +h, -d),
    (-w, +h, +d),

    # Face 5 do Cubo (baixo)
    (-w, -h, -d),
    (+w, -h, -d),         
    (-w, -h, +d),
    (+w, -h, +d),
    
    # Face 6 do Cubo (cima)
    (-w, +h, +d),
    (+w, +h, +d),           
    (-w, +h, -d),
    (+w, +h, -d)

]

### Para enviar nossos dados da CPU para a GPU, precisamos requisitar um slot (buffer).

In [12]:
# Request a buffer slot from GPU
buffer_VBO = glGenBuffers(1)
# Make this buffer the default one
glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO)


### Abaixo, nós enviamos todo o conteúdo da variável vertices.

Veja os parâmetros da função glBufferData [https://www.khronos.org/registry/OpenGL-Refpages/gl4/html/glBufferData.xhtml]

In [13]:
# Upload data
glBufferData(GL_ARRAY_BUFFER, vertices.nbytes, vertices, GL_DYNAMIC_DRAW)
glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO)

### Associando variáveis do programa GLSL (Vertex Shader) com nossos dados

Primeiro, definimos o byte inicial e o offset dos dados.

In [14]:
# Bind the position attribute
# --------------------------------------
stride = vertices.strides[0]
offset = ctypes.c_void_p(0)


Em seguida, soliciamos à GPU a localização da variável "position" (que guarda coordenadas dos nossos vértices). Nós definimos essa variável no Vertex Shader.

In [15]:
loc = glGetAttribLocation(program, "position")
glEnableVertexAttribArray(loc)

A partir da localização anterior, nós indicamos à GPU onde está o conteúdo (via posições stride/offset) para a variável position (aqui identificada na posição loc).

Outros parâmetros:

* Definimos que possui <b> três </b> coordenadas
* Que cada coordenada é do tipo float (GL_FLOAT)
* Que não se deve normalizar a coordenada (False)

Mais detalhes: https://www.khronos.org/registry/OpenGL-Refpages/gl4/html/glVertexAttribPointer.xhtml

In [16]:
glVertexAttribPointer(loc, 3, GL_FLOAT, False, stride, offset)

### Vamos pegar a localização da variável color para que possamos definir a cor em nosso laço da janela!

In [17]:
loc_color = glGetUniformLocation(program, "color")

### Capturando eventos de teclado e modificando variáveis para a matriz de transformação

### Nesse momento, nós exibimos a janela!


In [18]:
glfw.show_window(window)

# Programa principal

## Funções

In [19]:
def desenhar_assento():
    for i in range(7, 30, 4):
        glUniform4f(loc_color, 0.0, 0.0, 0.0, 1.0)
        glDrawArrays(GL_TRIANGLE_STRIP, i, i+4)

### Loop principal da janela.

In [20]:
angulo = 0

glEnable(GL_DEPTH_TEST) ### importante para 3D

def multiplica_matriz(a,b):
    m_a = a.reshape(4,4)
    m_b = b.reshape(4,4)
    m_c = np.dot(m_a,m_b)
    c = m_c.reshape(1,16)
    return c


while not glfw.window_should_close(window):
 
    #glPolygonMode(GL_FRONT_AND_BACK,GL_LINE)
        
    glClear(GL_COLOR_BUFFER_BIT | GL_DEPTH_BUFFER_BIT)    
    glClearColor(1.0, 1.0, 1.0, 1.0)

    mat_transform_comodo = np.array([1.0,  0.0, 0.0,     0.0, 
                              0.0,    1.0,   0.0, 0.0,
                              0.0,    0.0,   1.0, 0.0, 
                              0.0,    0.0,   0.0, 1.0], np.float32)
    
    loc_transformation = glGetUniformLocation(program, "mat_transformation")
    glUniformMatrix4fv(loc_transformation, 1, GL_TRUE, mat_transform_comodo) 
    
    glUniform4f(loc_color, 0.0, 0.0, 0.0, 1.0)
    glDrawArrays(GL_LINES, 0, 6)

    cos = math.cos(angulo)
    sin = math.sin(angulo)
    
    mat_rot_y = np.array([
        cos,  0.0, sin, 0.0,
        0.0,  1.0, 0.0, 0.0,
       -sin, 0.0, cos, 0.0,
        0.0,  0.0, 0.0, 1.0], np.float32)
    
    mat_t_assento = np.array([1.0,  0.0, 0.0, 0.5, 
                              0.0,    1.0,   0.0, 0.0,
                              0.0,    0.0,   1.0, 0.5, 
                              0.0,    0.0,   0.0, 1.0], np.float32)
    mat_transform_assento = multiplica_matriz(mat_t_assento, mat_rot_y)
        
    loc_transformation = glGetUniformLocation(program, "mat_transformation")
    glUniformMatrix4fv(loc_transformation, 1, GL_TRUE, mat_transform_assento) 
    
    #Desenhando cada face do assento
    desenhar_assento()
    
    glfw.swap_buffers(window)
    glfw.poll_events()

glfw.terminate()